# lenia-vlm on Colab: MaCE-Lenia runs

GPU runs for the live pipeline (see README.md). Results go to your Drive folder `lenia-runs/` and survive
disconnects; every run resumes if you re-run sections 1-2 and then the same command with the same `--out`.
The earlier 19-section notebook (classic Lenia, distillation, hill climber, MAP-Elites v1-v4) is in
`archive/colab/lenia_colab.ipynb`.

## 1. GPU and Drive

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime -> Change runtime type -> GPU"
import os
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/lenia-runs'      # every result goes here and survives disconnects
os.makedirs(RUNS, exist_ok=True)
print('results folder:', RUNS)

## 2. Code, dependencies, judge keys

In [ ]:
import os, subprocess
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

REPO, DIR = 'github.com/feli10/lenia-vlm.git', '/content/lenia-vlm'
tok = secret('GITHUB_TOKEN')
url = f'https://x-access-token:{tok}@{REPO}' if tok else f'https://{REPO}'
if not os.path.isdir(DIR):
    r = subprocess.run(['git', 'clone', '-q', url, DIR], capture_output=True, text=True)
else:
    r = subprocess.run(['git', '-C', DIR, 'pull', '-q', url, 'main'], capture_output=True, text=True)
print('git:', 'ok' if r.returncode == 0 else r.stderr.replace(tok or '\0', '***'))
subprocess.run(['git', '-C', DIR, 'remote', 'set-url', 'origin', f'https://{REPO}'])   # don't keep the token in .git
os.chdir(DIR)
!git log --oneline -1
!pip install -q -e .

In [ ]:
# API keys for the judges, if you added them as Colab secrets
keys = {k: secret(k) for k in ('OPENROUTER_API_KEY', 'TYPESAFE_API_KEY', 'GOOGLE_CLOUD_PROJECT')}
with open('.env', 'w') as f:
    for k, v in keys.items():
        if v:
            f.write(f'{k}={v}\n')
print('judge keys present:', [k for k, v in keys.items() if v] or 'none (fine for simulation-only runs)')

## 3. Checks

The test suite, offline: the golden test (prompts, pictures, cache keys, mutation and simulation unchanged, so
old caches and runs stay valid) and the PyTorch-vs-numpy parity tests. Do not use the GPU for runs unless these
pass.

In [ ]:
!pip install -q pytest && python -m pytest -q tests

## 4. Base search: MAP-Elites + surprise archive

The command that produced `mace-qd5-sarch-s0`, the seed 0 base run the stage-1 core test reuses (4 islands,
bodies x form x motion, complexity entry duel, surprise archive). About 2.3 h and ~$3 for 100 iterations.
Kept for reference; the seed 1 base run for stage 1 (shorter, with the record logging) is added here when it
is built.

In [ ]:
# reference: seed 0 (already done)
# !cd /content/lenia-vlm && python scripts/mace_qd.py run --backend auto --islands 4 --axes bodies,form,motion --body-edges 3,9 --motion-edges 0.75,1.0,1.25 --iterations 100 --sarch --seed 0 --out {RUNS}/mace-qd5-sarch-s0

## 5. Stage 1: expeditions (seed 0)

Each arm runs one 10-step expedition (32 mutants a step, judged against the trigger's start: expedition v2) from each of the 12 triggers in `stage1-s0/triggers.json`.
Copy the Mac's `runs/stage1-s0` folder to Drive `lenia-runs/stage1-s0` first (triggers, pictures, the shared judge and
VLM caches). Run the arms in order: **L** (then `check`: gate 0c, the floor), then **N**, **NR**, **NS**. NR and NS
reuse N's first 5 steps from the caches and ask Gemini Pro for one review per trigger, which needs the Gemini cell
below (Vertex AI, billed to your Google Cloud project; add `GOOGLE_CLOUD_PROJECT` as a Colab secret). Every command
resumes where it stopped.

Results go to `stage1-s0/exp-v2/<arm>/`. (`stage1-s0/arms/L` is the first, v1 run of arm L, kept as the record of why v1 was replaced.)

In [ ]:
# Gemini on Vertex (arms NR and NS; arm N's names are already cached): sign in once per session
!pip install -q google-genai
from google.colab import auth
auth.authenticate_user()

In [ ]:
S1, BASE = f'{RUNS}/stage1-s0', f'{RUNS}/mace-qd5-sarch-s0'
!cd /content/lenia-vlm && python scripts/stage1.py expedition --arm L --run {BASE} --out {S1} --backend auto

In [ ]:
# held-out Luna, end vs start (gate 0c for arm L); no GPU needed, can also run on the Mac
!cd /content/lenia-vlm && python scripts/stage1.py check --arm L --run {BASE} --out {S1}

In [ ]:
# after gate 0c passes and Felix has looked at exp-v2/L/check.png:
# !cd /content/lenia-vlm && python scripts/stage1.py expedition --arm N  --run {BASE} --out {S1} --backend auto
# !cd /content/lenia-vlm && python scripts/stage1.py expedition --arm NR --run {BASE} --out {S1} --backend auto
# !cd /content/lenia-vlm && python scripts/stage1.py expedition --arm NS --run {BASE} --out {S1} --backend auto
# !cd /content/lenia-vlm && for A in N NR NS; do python scripts/stage1.py check --arm $A --run {BASE} --out {S1}; done

## Notes

- The free tier disconnects idle sessions and caps sessions at about 12 hours. After a disconnect the machine
  is wiped, your Drive is not: re-run sections 1-2, then the same command with the same `--out`.
- Judge calls made here are cached in the run folder (`judge_cache.sqlite`), so a resumed run does not pay twice.
- Claude can read your Drive folder: say when a run has finished.